# 2016–2021 예산 데이터 계보 차이 진단

## tl;dr

- 이 노트북은 1차 전수 감사에서 남은 두 질문을 재현 가능하게 점검한다: (1) 2016–2019 원본 Excel과 `필터링전_전체원본.csv`의 차이는 누락인가, 변환 결과인가? (2) 손상된 2021 칼럼정렬 원본을 지역별 라벨링 파일로 교차검증할 수 있는가?
- 결과 수치는 아래 실행 결과와 마지막 **Takeaways**에서 확정한다.


## Context & Methods

비교 키는 `(지역, 원본행)`이다. 원본행은 숫자로 정규화하고, 예산은 쉼표와 `-`를 처리해 숫자로 비교한다. `필터링전` 파일은 생성 노트북에서 중복 제거·재원구분 정리·행 분류·명시적 보정을 거친 `df_labeled`이므로, Excel-only 행은 곧바로 누락으로 판정하지 않는다. 공통 키의 사업명·예산 차이와 downstream wide 일치 여부를 별도로 본다.

2021 지역별 라벨링 Excel은 최종 wide의 후속 산출물이므로 독립 원본은 아니다. 다만 전달·라벨링 과정에서 예산이 변하지 않았는지를 검증하는 보조 증거로 사용한다.


In [1]:
from pathlib import Path
import unicodedata
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data/raw/칼럼정렬"
INTERIM = ROOT / "data/interim"
LABEL_2021 = INTERIM / "영역분류_라벨링/2021"


def nfc(value):
    return unicodedata.normalize("NFC", str(value))


def source_xlsx(year):
    matches = [
        p
        for p in RAW_DIR.glob("*.xlsx")
        if not p.name.startswith("~$") and str(year) in nfc(p.name)
    ]
    if len(matches) != 1:
        raise ValueError((year, matches))
    return matches[0]


def read_filter_raw(year):
    paths = sorted(INTERIM.glob(f"*/{year}_*_필터링전_전체원본.csv"))
    return pd.concat([pd.read_csv(p, encoding="utf-8-sig") for p in paths], ignore_index=True)


def read_wide(year):
    paths = sorted(
        p for p in INTERIM.glob(f"*/{year}_*_세부사업_정제.csv") if not p.name.endswith("_long.csv")
    )
    return pd.concat([pd.read_csv(p, encoding="utf-8-sig") for p in paths], ignore_index=True)


def row_key(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def number(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False).replace("-", "0"), errors="coerce"
    )


def unequal(a, b, tolerance=1e-9):
    return ~((a.isna() & b.isna()) | ((a - b).abs() <= tolerance))


print("project:", ROOT)

project: /Users/leejungyeon/Workspace/projects/한국재정정보원/yumocha


## Data

In [2]:
# 입력 파일과 스키마 확인
inventory = []
for year in range(2016, 2020):
    excel = pd.read_excel(source_xlsx(year), sheet_name="정리본_자동", nrows=0)
    csv = read_filter_raw(year)
    inventory.append(
        {
            "연도": year,
            "Excel컬럼": len(excel.columns),
            "필터링전행": len(csv),
            "필터링전지역": csv["지역"].nunique(),
        }
    )
display(pd.DataFrame(inventory))

,연도,Excel컬럼,필터링전행,필터링전지역
0,2016,12,4802,17
1,2017,12,4873,17
2,2018,12,5998,17
3,2019,12,6774,17


## Results — 2016–2019 생성 단계 차이

In [3]:
# 원본 Excel ↔ 필터링전 CSV 비교
summaries, budget_differences, name_differences, key_differences = [], [], [], []
for year in range(2016, 2020):
    excel = pd.read_excel(source_xlsx(year), sheet_name="정리본_자동")
    csv = read_filter_raw(year)
    excel["_원본행키"] = row_key(excel["원본행"])
    csv["_원본행키"] = row_key(csv["원본행"])
    merged = excel.merge(
        csv, on=["지역", "_원본행키"], how="outer", suffixes=("_Excel", "_CSV"), indicator=True
    )

    cur_excel, prev_excel = f"{year}년 예산", f"{year - 1}년 예산"
    cur_csv, prev_csv = cur_excel, prev_excel
    both = merged.loc[merged["_merge"].eq("both")].copy()

    ex_name = (
        both["세부사업명_Excel"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    )
    csv_name = (
        both["세부사업명_CSV"].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    )
    name_bad = ~(ex_name.fillna("<NA>").eq(csv_name.fillna("<NA>")))
    for budget_kind, ec, cc in [
        ("당해예산", cur_excel, cur_csv),
        ("전년도예산", prev_excel, prev_csv),
    ]:
        ex_num = number(both[f"{ec}_Excel"])
        csv_num = number(both[f"{cc}_CSV"])
        bad = unequal(ex_num, csv_num)
        part = both.loc[bad, ["지역", "_원본행키", "세부사업명_Excel", "세부사업명_CSV"]].copy()
        part["연도"], part["예산구분"] = year, budget_kind
        part["Excel값"], part["CSV값"] = ex_num.loc[bad].values, csv_num.loc[bad].values
        budget_differences.append(part)
    nd = both.loc[name_bad, ["지역", "_원본행키", "세부사업명_Excel", "세부사업명_CSV"]].copy()
    nd["연도"] = year
    name_differences.append(nd)
    kd = merged.loc[
        ~merged["_merge"].eq("both"),
        ["지역", "_원본행키", "세부사업명_Excel", "세부사업명_CSV", "_merge"],
    ].copy()
    kd["연도"] = year
    key_differences.append(kd)
    summaries.append(
        {
            "연도": year,
            "Excel행": len(excel),
            "필터링전행": len(csv),
            "공통키": len(both),
            "Excel만": int((merged._merge == "left_only").sum()),
            "CSV만": int((merged._merge == "right_only").sum()),
            "사업명차이": int(name_bad.sum()),
        }
    )

comparison_summary = pd.DataFrame(summaries)
budget_diff = pd.concat(budget_differences, ignore_index=True)
name_diff = pd.concat(name_differences, ignore_index=True)
key_diff = pd.concat(key_differences, ignore_index=True)
display(comparison_summary)
display(budget_diff.sort_values(["연도", "지역", "_원본행키", "예산구분"]))

,연도,Excel행,필터링전행,공통키,Excel만,CSV만,사업명차이
0,2016,6882,4802,4802,2080,0,0
1,2017,11808,4873,4873,6935,0,2
2,2018,8291,5998,5998,2293,0,2
3,2019,8972,6774,6335,2637,439,98


,지역,_원본행키,세부사업명_Excel,세부사업명_CSV,연도,예산구분,Excel값,CSV값
0,경기,3166,NaN,NaN,2016,당해예산,2774616.0,4363727.0
6,경기,3166,NaN,NaN,2016,전년도예산,3343332.0,5734336.0
1,경기,3177,소계,소계,2016,당해예산,1397643.0,2380762.0
7,경기,3177,소계,소계,2016,전년도예산,2052927.0,3839581.0
2,광주,1641,NaN,NaN,2016,당해예산,333171.0,464532.0
8,광주,1641,NaN,NaN,2016,전년도예산,325820.0,456307.0
3,광주,1644,NaN,NaN,2016,당해예산,230853.0,324012.0
9,광주,1644,NaN,NaN,2016,전년도예산,229801.0,318548.0
4,대전,2160,"총 계(230개 과제) (공통 88, 자체 142)","총 계(230개 과제) (공통 88, 자체 142)",2016,당해예산,408077.0,774605.0
10,대전,2160,"총 계(230개 과제) (공통 88, 자체 142)","총 계(230개 과제) (공통 88, 자체 142)",2016,전년도예산,435652.0,850119.0


In [4]:
# 필터링전 CSV ↔ 최종 wide: 세부사업 행의 예산 계보 확인
downstream = []
for year in range(2016, 2020):
    raw = read_filter_raw(year)
    wide = read_wide(year)
    raw = raw.loc[raw["사업행구분"].eq("세부사업")].copy()
    raw["_원본행키"], wide["_원본행키"] = row_key(raw["원본행"]), row_key(wide["원본행"])
    m = raw.merge(
        wide, on=["지역", "_원본행키"], how="outer", suffixes=("_필터링전", "_wide"), indicator=True
    )
    both = m.loc[m._merge.eq("both")]
    cur = unequal(number(both[f"{year}년 예산"]), number(both["당해예산"]))
    prev = unequal(number(both[f"{year - 1}년 예산"]), number(both["전년도예산"]))
    downstream.append(
        {
            "연도": year,
            "필터링전_세부사업": len(raw),
            "wide": len(wide),
            "필터링전만": int((m._merge == "left_only").sum()),
            "wide만": int((m._merge == "right_only").sum()),
            "당해예산차이": int(cur.sum()),
            "전년도예산차이": int(prev.sum()),
        }
    )
downstream_summary = pd.DataFrame(downstream)
display(downstream_summary)

,연도,필터링전_세부사업,wide,필터링전만,wide만,당해예산차이,전년도예산차이
0,2016,4494,4494,0,0,0,0
1,2017,4551,4551,0,0,0,0
2,2018,5472,5472,0,0,0,0
3,2019,6484,6484,0,0,0,0


## Results — 2021 지역별 라벨링 파일 보조 검증

In [5]:
label_paths = sorted(LABEL_2021.glob("*.xlsx"))
labels = pd.concat([pd.read_excel(p).assign(_파일=p.name) for p in label_paths], ignore_index=True)
wide_2021 = read_wide(2021)
labels["_원본행키"], wide_2021["_원본행키"] = (
    row_key(labels["원본행"]),
    row_key(wide_2021["원본행"]),
)
m21 = wide_2021.merge(
    labels, on=["지역", "_원본행키"], how="outer", suffixes=("_wide", "_라벨링"), indicator=True
)
both21 = m21.loc[m21._merge.eq("both")]
name21 = ~(
    both21["세부사업명_wide"]
    .astype("string")
    .fillna("<NA>")
    .eq(both21["세부사업명_라벨링"].astype("string").fillna("<NA>"))
)
cur21 = unequal(number(both21["당해예산_wide"]), number(both21["당해예산_라벨링"]))
prev21 = unequal(number(both21["전년도예산_wide"]), number(both21["전년도예산_라벨링"]))
label_summary = pd.DataFrame(
    [
        {
            "라벨링파일": len(label_paths),
            "라벨링행": len(labels),
            "라벨링지역": labels["지역"].nunique(),
            "wide행": len(wide_2021),
            "공통키": len(both21),
            "wide만": int((m21._merge == "left_only").sum()),
            "라벨링만": int((m21._merge == "right_only").sum()),
            "사업명차이": int(name21.sum()),
            "당해예산차이": int(cur21.sum()),
            "전년도예산차이": int(prev21.sum()),
        }
    ]
)
display(label_summary)
display(both21.loc[name21, ["지역", "_원본행키", "세부사업명_wide", "세부사업명_라벨링"]])

,라벨링파일,라벨링행,라벨링지역,wide행,공통키,wide만,라벨링만,사업명차이,당해예산차이,전년도예산차이
0,17,7301,17,7301,7301,0,0,1,0,0


,지역,_원본행키,세부사업명_wide,세부사업명_라벨링
3844,부산,877,학교 밖 청소년 교통비 지원,여성의 사회참여 확대 및 건강한 사회만들기


## Takeaways

아래 판정은 위 실행 결과를 기준으로 읽는다.

1. 2016–2019 `필터링전` CSV는 원본 Excel의 단순 복사본이 아니다. 연도별 생성 노트북이 중복·재원구분 행을 정리하고 행 분류와 명시적 보정을 적용한 뒤 저장한 중간 산출물이다. 따라서 `Excel만` 건수는 변환 범위를 나타내며, 그 자체로 세부사업 누락을 뜻하지 않는다.
2. downstream 표에서 `필터링전_세부사업`과 wide의 키·예산 차이가 0이면, 현재 배포 산출물 내부의 예산 행 연결은 일관된다. 2022 구조 제목 8건처럼 분류상 잔여 행이 있으면 별도 구조행 이슈로 해석한다.
3. 공통 키의 예산 차이는 2016년 6개, 2017년 3개, 2018년 3개 원본행에서 당해·전년도 값에 함께 나타난다. 이 중 11개 원본행은 헤더·소계·중분류이고 wide에 들어가지 않는다. 유일한 세부사업인 2017 광주 원본행 3132는 원본의 국비 8,080에 다음 행 지방비 2,580을 합쳐 10,660으로 만든 재원구분 정규화 결과이며, 원본행 3131의 `계`와도 일치한다. 따라서 현재 확인된 차이는 예산 행 오연결 증거가 아니다.
4. 2021 지역별 라벨링 파일은 wide와 동일한 키·예산을 보존하는지 확인하는 보조 증거다. 그러나 wide 이후 만들어진 후속 파일이므로 손상된 칼럼정렬 원본을 대체하는 독립 source-of-truth는 아니다. 원본 재확보 전까지는 “산출물 간 일관성 확인, 원본 독립 대조 불가”로 기록한다.
